<a href="https://colab.research.google.com/github/MasAz1/MasAz1/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Training Model Logistic Regresion dengan Polynomial**

In [ ]:
# =========================================================
# TRAINING MODEL DETEKSI KEBOCORAN PIPA
# Logistic Regression Dengan Polynomial
# =========================================================

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score

# =========================================================
# 1. MEMUAT DATASET
# =========================================================

df_orig = pd.read_csv("location_aware_gis_leakage_dataset.csv")[["Pressure", "Flow_Rate", "Leakage_Flag"]]
df_aug1 = pd.read_csv("augmentasi_1.csv")[["Pressure", "Flow_Rate", "Leakage_Flag"]]
df_aug2 = pd.read_csv("augmentasi_2.csv")[["Pressure", "Flow_Rate", "Leakage_Flag"]]

df = pd.concat(
    [df_orig, df_aug1, df_aug2],
    ignore_index=True
)

print("======================================")
print(" INFORMASI DATASET AWAL")
print("======================================")
print(f"Total data awal: {len(df)}")
print("\nDistribusi label awal:")
print(df["Leakage_Flag"].value_counts())

# =========================================================
# VISUALISASI INFORMASI DATASET AWAL
# =========================================================

label_awal = df["Leakage_Flag"].value_counts().sort_index()

df_label_awal = pd.DataFrame({
    "Status": ["Normal", "Bocor"],
    "Jumlah": [
        label_awal.get(0, 0),
        label_awal.get(1, 0)
    ]
})

plt.figure(figsize=(6, 4))

sns.barplot(
    data=df_label_awal,
    x="Status",
    y="Jumlah"
)

plt.title("Informasi Dataset Awal - Distribusi Label")
plt.xlabel("Status Kebocoran")
plt.ylabel("Jumlah Data")

for index, row in df_label_awal.iterrows():
    plt.text(
        index,
        row["Jumlah"],
        f'{row["Jumlah"]:,}'.replace(",", "."),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# =========================================================
# 2. PEMBERSIHAN DATA
# =========================================================

# Menghapus data kosong
df = df.dropna()

# Menghapus data duplikat
df = df.drop_duplicates()

# Pastikan tipe data numerik
df["Pressure"] = pd.to_numeric(df["Pressure"], errors="coerce")
df["Flow_Rate"] = pd.to_numeric(df["Flow_Rate"], errors="coerce")
df["Leakage_Flag"] = pd.to_numeric(df["Leakage_Flag"], errors="coerce")

df = df.dropna()

# Menghapus outlier menggunakan Z-Score
z_scores = np.abs(stats.zscore(df[["Pressure", "Flow_Rate"]]))
df = df[(z_scores < 3).all(axis=1)]

# Smoothing menggunakan Moving Average
df["Pressure"] = df["Pressure"].rolling(window=3, min_periods=1).mean()
df["Flow_Rate"] = df["Flow_Rate"].rolling(window=3, min_periods=1).mean()

# Cleaning ulang setelah smoothing
df = df.dropna().drop_duplicates()

print("\n======================================")
print(" INFORMASI DATASET SETELAH CLEANING")
print("======================================")
print(f"Total data setelah dibersihkan: {len(df)}")
print("\nDistribusi label setelah cleaning:")
print(df["Leakage_Flag"].value_counts())

# =========================================================
# VISUALISASI INFORMASI DATASET SETELAH CLEANING
# =========================================================

label_cleaning = df["Leakage_Flag"].value_counts().sort_index()

total_cleaning = len(df)
data_normal = label_cleaning.get(0, 0)
data_bocor = label_cleaning.get(1, 0)

df_info_cleaning = pd.DataFrame({
    "Keterangan": [
        "Total Cleaning",
        "Data Normal",
        "Data Bocor"
    ],
    "Jumlah Data": [
        total_cleaning,
        data_normal,
        data_bocor
    ]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=df_info_cleaning,
    x="Keterangan",
    y="Jumlah Data"
)

plt.title("Informasi Dataset Setelah Cleaning")
plt.xlabel("Keterangan")
plt.ylabel("Jumlah Data")

for index, row in df_info_cleaning.iterrows():
    plt.text(
        index,
        row["Jumlah Data"],
        f'{row["Jumlah Data"]:,}'.replace(",", "."),
        ha="center",
        va="bottom"
    )

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nTabel Informasi Dataset Setelah Cleaning:")
print(df_info_cleaning.to_string(index=False))

# =========================================================
# 3. PEMISAHAN FITUR DAN TARGET
# =========================================================

X = df[["Pressure", "Flow_Rate"]]
y = df["Leakage_Flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n======================================")
print(" DATA TRAINING DAN TESTING")
print("======================================")
print(f"Jumlah data training: {len(X_train)}")
print(f"Jumlah data testing : {len(X_test)}")

print("\nDistribusi label training:")
print(y_train.value_counts())

print("\nDistribusi label testing:")
print(y_test.value_counts())

# =========================================================
# VISUALISASI PEMBAGIAN DATA TRAINING DAN TESTING
# =========================================================

jumlah_data = pd.DataFrame({
    "Jenis Data": ["Training", "Testing"],
    "Jumlah": [len(X_train), len(X_test)]
})

plt.figure(figsize=(6, 4))

sns.barplot(
    data=jumlah_data,
    x="Jenis Data",
    y="Jumlah"
)

plt.title("Pembagian Data Training dan Testing")
plt.xlabel("Jenis Data")
plt.ylabel("Jumlah Data")

for index, row in jumlah_data.iterrows():
    plt.text(
        index,
        row["Jumlah"],
        f'{row["Jumlah"]:,}'.replace(",", "."),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# =========================================================
# VISUALISASI DISTRIBUSI LABEL DATA TRAINING
# =========================================================

label_training = y_train.value_counts().sort_index()

df_train_visual = pd.DataFrame({
    "Status": ["Normal", "Bocor"],
    "Jumlah": [
        label_training.get(0, 0),
        label_training.get(1, 0)
    ]
})

plt.figure(figsize=(6, 4))

sns.barplot(
    data=df_train_visual,
    x="Status",
    y="Jumlah"
)

plt.title("Distribusi Label Data Training")
plt.xlabel("Status Kebocoran")
plt.ylabel("Jumlah Data")

for index, row in df_train_visual.iterrows():
    plt.text(
        index,
        row["Jumlah"],
        f'{row["Jumlah"]:,}'.replace(",", "."),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# =========================================================
# VISUALISASI DISTRIBUSI LABEL DATA TESTING
# =========================================================

label_testing = y_test.value_counts().sort_index()

df_test_visual = pd.DataFrame({
    "Status": ["Normal", "Bocor"],
    "Jumlah": [
        label_testing.get(0, 0),
        label_testing.get(1, 0)
    ]
})

plt.figure(figsize=(6, 4))

sns.barplot(
    data=df_test_visual,
    x="Status",
    y="Jumlah"
)

plt.title("Distribusi Label Data Testing")
plt.xlabel("Status Kebocoran")
plt.ylabel("Jumlah Data")

for index, row in df_test_visual.iterrows():
    plt.text(
        index,
        row["Jumlah"],
        f'{row["Jumlah"]:,}'.replace(",", "."),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# =========================================================
# 4. MEMBUAT MODEL LR POLY
# =========================================================
# Pipeline ini membuat semua proses masuk ke 1 model:
# PolynomialFeatures -> StandardScaler -> SMOTE -> Logistic Regression

model_lr_poly = Pipeline([
    ("poly", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(random_state=42))
])

# =========================================================
# 5. PARAMETER GRIDSEARCHCV
# =========================================================

param_grid = {
    "poly__degree": [2, 3],
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"],
    "model__max_iter": [1000, 2000, 3000]
}

# =========================================================
# 6. TRAINING DENGAN GRIDSEARCHCV
# =========================================================

grid_search = GridSearchCV(
    estimator=model_lr_poly,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("\n======================================")
print(" PROSES TRAINING GRIDSEARCHCV")
print("======================================")

grid_search.fit(X_train, y_train)

print("\n[INFO] Training selesai.")

# Ambil model terbaik
best_model = grid_search.best_estimator_

print("\n======================================")
print(" HASIL PARAMETER TERBAIK")
print("======================================")
print(grid_search.best_params_)
print(f"Akurasi Cross Validation terbaik: {grid_search.best_score_ * 100:.2f}%")

# =========================================================
# 7. EVALUASI MODEL
# =========================================================

y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("\n======================================")
print(" HASIL EVALUASI MODEL")
print("======================================")
print(f"Accuracy Testing: {acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================================================
# 8. ROC CURVE + CONFUSION MATRIX BERDAMPINGAN
# =========================================================

# Probabilitas prediksi untuk kelas 1 (Bocor)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Hitung ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

print("\n======================================")
print(" ROC AUC SCORE")
print("======================================")
print(f"ROC AUC Score: {auc_score:.4f}")

# Hitung confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# -------------------------
# Plot ROC Curve
# -------------------------

ax[0].plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.4f})")
ax[0].plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
ax[0].set_title("ROC Curve - Logistic Regression + Polynomial")
ax[0].set_xlabel("False Positive Rate")
ax[0].set_ylabel("True Positive Rate")
ax[0].legend()
ax[0].grid(True)

# -------------------------
# Plot Confusion Matrix
# -------------------------

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Normal", "Bocor"],
    yticklabels=["Normal", "Bocor"],
    ax=ax[1]
)

ax[1].set_title("Confusion Matrix")
ax[1].set_xlabel("Prediksi")
ax[1].set_ylabel("Aktual")

plt.tight_layout()
plt.show()

# =========================================================
# 9. SIMPAN MODEL
# =========================================================

joblib.dump(best_model, "logisticregresion.pkl")

**Testing model**

In [ ]:
# =========================================================
# TESTING MODEL DETEKSI KEBOCORAN PIPA
# Model file: logisticregresion.pkl
# =========================================================

import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# =========================================================
# 1. LOAD MODEL
# =========================================================

model = joblib.load("logisticregresion.pkl")

print("========================================")
print(" MODEL BERHASIL DIMUAT")
print("========================================")
print("File model: logisticregresion.pkl")

# =========================================================
# 2. LOAD DATA TESTING
# =========================================================

df = pd.read_csv("testing_mix_60_training_40_sintetis_4000.csv")

df = df[["Pressure", "Flow_Rate", "Leakage_Flag"]]

print("\n========================================")
print(" DATA TESTING")
print("========================================")
print("Total data awal:", len(df))
print(df.head())

print("\nDistribusi label awal:")
print(df["Leakage_Flag"].value_counts())

# =========================================================
# 3. CLEANING DATA
# =========================================================

df = df.dropna().drop_duplicates()

df["Pressure"] = pd.to_numeric(df["Pressure"], errors="coerce")
df["Flow_Rate"] = pd.to_numeric(df["Flow_Rate"], errors="coerce")
df["Leakage_Flag"] = pd.to_numeric(df["Leakage_Flag"], errors="coerce")

df = df.dropna()

print("\n========================================")
print(" DATA SETELAH CLEANING")
print("========================================")
print("Total data setelah cleaning:", len(df))

print("\nDistribusi label setelah cleaning:")
print(df["Leakage_Flag"].value_counts())

# =========================================================
# 4. PISAHKAN FITUR DAN TARGET
# =========================================================

X = df[["Pressure", "Flow_Rate"]]
y = df["Leakage_Flag"]

# =========================================================
# 5. PREDIKSI
# =========================================================
# Karena model sudah 1 file pipeline,
# tidak perlu poly.transform dan scaler.transform

y_pred = model.predict(X)

df["Prediksi"] = y_pred

# =========================================================
# 6. EVALUASI MODEL
# =========================================================

acc = accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred, labels=[0, 1])

print("\n========================================")
print(" HASIL TESTING MODEL")
print("========================================")
print(f"Accuracy: {acc * 100:.2f}%")

print("\nConfusion Matrix:")
print(cm)

print("\nKeterangan Confusion Matrix:")
print("TN / Normal diprediksi Normal :", cm[0][0])
print("FP / Normal diprediksi Bocor  :", cm[0][1])
print("FN / Bocor diprediksi Normal  :", cm[1][0])
print("TP / Bocor diprediksi Bocor   :", cm[1][1])

print("\nClassification Report:")
print(classification_report(
    y,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Bocor"]
))

# =========================================================
# 7. VISUALISASI CONFUSION MATRIX
# =========================================================

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Normal", "Bocor"],
    yticklabels=["Normal", "Bocor"]
)

plt.title("Confusion Matrix - Testing Logistic Regression")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

# =========================================================
# 8. UBAH LABEL ANGKA KE STATUS
# =========================================================

df["Status_Aktual"] = df["Leakage_Flag"].map({
    0: "Normal",
    1: "Bocor"
})

df["Status_Prediksi"] = df["Prediksi"].map({
    0: "Normal",
    1: "Bocor"
})

# =========================================================
# 9. SIMPAN HASIL TESTING
# =========================================================

df.to_csv("hasil_testing_logreg_poly.csv", index=False)

print("\n========================================")
print(" HASIL TESTING DISIMPAN")
print("========================================")
print("File: hasil_testing_logreg_poly.csv")